# IOH Outcome Labeling

**Goal**: Assign a binary hypotension label to every episode in
`arrhythmia_episodes_updated.csv`.  The outcome is defined as MAP < 65 mmHg
occurring at any point in the **5 minutes immediately after `episode_end_sec`**.

**BP source per episode**:
- `ART_MBP` patients: load `Solar8000/ART_MBP` (continuous, ~1 Hz)
- `NIBP_estimated` patients: load `Solar8000/NIBP_SBP` + `NIBP_DBP`, compute
  `estimated_MAP = DBP + (SBP − DBP) / 3`
- Episodes with `nibp_outcome_imputed = True`: the NIBP_data stage already
  confirmed a reading exists in the peri-episode window; fall back to that
  window if the outcome window is empty (these episodes **cannot be NaN**).

**Output**: `IOH_labels/episode_hypotension_labels.csv`

In [11]:
import numpy as np
import pandas as pd
import vitaldb
from pathlib import Path
import random

print('Libraries loaded.')

Libraries loaded.


In [ ]:
# Paths
BASE_DIR       = Path('..').resolve()          # project root
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR     = DATA_PROCESSED

EPISODES_CSV = DATA_PROCESSED / 'arrhythmia_episodes_updated.csv'

# VitalDB track names
ART_TRACK      = 'Solar8000/ART_MBP'
NIBP_SBP_TRACK = 'Solar8000/NIBP_SBP'
NIBP_DBP_TRACK = 'Solar8000/NIBP_DBP'
INTERVAL_SEC   = 1.0    # resample everything to 1-second grid

# ART_MBP artifact cleaning
ART_MIN, ART_MAX = 40, 180          # mmHg — physiological plausibility range
MAP_MIN,  MAP_MAX = 40, 180         # alias names used in comments/docs for clarity
STABLE_RUN_SEC   = 10               # consecutive valid seconds needed before
                                     # we consider the arterial line zeroed and stable

# NIBP artifact cleaning (same thresholds as the NIBP-source-labeling stage)
NIBP_SBP_MIN, NIBP_SBP_MAX = 60, 220
NIBP_DBP_MIN, NIBP_DBP_MAX = 30, 130
NIBP_MAP_MIN, NIBP_MAP_MAX = 40, 150

# Heart rate and SpO2 plausibility bounds (used in later feature stages)
HR_MIN   = 20    # bpm — below this is artifact or asystole
HR_MAX   = 250   # bpm — above this is artifact
SPO2_MIN = 85    # % — below this is artifact or severe desaturation
SPO2_MAX = 100   # % — maximum physically possible

# Outcome window
OUTCOME_WINDOW_SEC = 300            # 5 minutes after episode_end_sec

# Labeling
HYPO_THRESHOLD     = 65             # MAP < 65 mmHg = hypotension
MIN_VALID_READINGS = 10             # fewer than this → NaN (for non-imputed episodes)
MIN_VALID_VALUES   = MIN_VALID_READINGS  # alias — both names refer to the same threshold

# Imputed NIBP fallback window (identical to the NIBP-source-labeling stage)
NIBP_EXPANDED_BEFORE = 300          # seconds before episode_start_sec
NIBP_EXPANDED_AFTER  = 600          # seconds after  episode_start_sec

print('Constants set.')
print(f'  Input : {EPISODES_CSV}')
print(f'  Output: {OUTPUT_DIR.resolve()}')

In [13]:
episodes = pd.read_csv(EPISODES_CSV)

# nibp_outcome_imputed comes back as bool from this CSV; confirm.
print(f'Loaded {len(episodes)} episodes from {episodes["caseid"].nunique()} patients')
print(f'Columns: {episodes.columns.tolist()}')
print()
print('bp_source:')
print(episodes['bp_source'].value_counts().to_string())
print()
print('nibp_outcome_imputed:')
print(episodes['nibp_outcome_imputed'].value_counts().to_string())

Loaded 1284 episodes from 457 patients
Columns: ['caseid', 'episode_number', 'episode_start_sec', 'episode_end_sec', 'episode_duration_sec', 'episode_beat_count', 'episode_dominant_rhythm', 'episode_beat_type', 'episode_rr_cv', 'bp_source', 'nibp_outcome_imputed']

bp_source:
bp_source
ART_MBP           982
NIBP_estimated    302

nibp_outcome_imputed:
nibp_outcome_imputed
False    1282
True        2


In [14]:
def find_stable_start(arr, window=10, vmin=40, vmax=180):
    """
    Return the index where ART_MBP becomes reliably stable — i.e., where the
    recording has exited the pre-zeroing artifact phase and the arterial line is
    properly placed and reading physiological values.

    Everything BEFORE this index gets set to NaN in the calling code.

    Returns len(arr) if no stable region is found (whole array gets NaN'd).

    ── WHY THE ORIGINAL ALGORITHM BROKE ────────────────────────────────────────
    The original version counted STRICTLY CONSECUTIVE seconds where the value
    was valid (not NaN and within [vmin, vmax]):

        consecutive += 1
        if consecutive >= window:   # required 10 consecutive valid seconds
            return i - window + 1

    This worked on the synthetic unit-test array (3 zeros then 10 valid readings
    all in a row). BUT real ART_MBP from VitalDB's Solar8000 monitor produces an
    ALTERNATING NaN/value pattern at 1-second resolution because the monitor
    only refreshes every ~1-2 seconds. The signal looks like:

        [..., NaN, 78.0, NaN, 81.0, NaN, 76.0, NaN, 80.0, ...]

    With this alternating pattern, `consecutive` never exceeded 1 — every other
    second reset it to 0. So the function always returned len(arr), meaning
    "no stable section found." The calling code then executed:

        art[:len(arr)] = np.nan   →   the ENTIRE signal was wiped

    This silently zeroed every patient's ART_MBP array, leaving 0 valid readings
    in every outcome window and producing hypotension_label = NaN for all ~982
    ART_MBP episodes (seen as 986 total NaN labels in the output).

    ── WHAT CHANGED ────────────────────────────────────────────────────────────
    Instead of requiring `window` CONSECUTIVE valid seconds, we now use a DENSITY
    CHECK over a wider span:

      1. Anchor on the first second that itself holds a valid reading.
      2. Look ahead over the next (window * 2) seconds.
      3. If at least `window` of those seconds are valid → return this index.

    Using (window * 2) = 20 seconds accounts for the alternating pattern:
    in any 20-second stretch of alternating NaN/value there are ~10 valid readings,
    which equals the required `window`. Pre-zeroing sections (all NaN after Step 1
    artifact cleaning) never produce a valid anchor, so the function still returns
    len(arr) correctly for those patients.
    """
    # check_span is twice the window to accommodate the alternating NaN/value
    # pattern of Solar8000 numeric tracks at 1-second resolution.
    check_span = window * 2

    for i in range(len(arr)):
        v = arr[i]

        # The anchor second must itself hold a valid reading.
        # Out-of-range or NaN values (already set to NaN by Step 1 cleaning) are skipped.
        # This ensures we never anchor on a pre-zeroing zero or an artifact spike.
        if np.isnan(v) or v < vmin or v > vmax:
            continue

        # Count how many of the next check_span seconds have valid readings.
        span_end = min(len(arr), i + check_span)
        segment  = arr[i:span_end]
        n_valid  = int(np.sum(~np.isnan(segment) & (segment >= vmin) & (segment <= vmax)))

        # If at least `window` of them are valid, this is the start of stable data.
        if n_valid >= window:
            return i

    return len(arr)   # no stable region found — calling code will wipe the whole array


# ── Unit test ────────────────────────────────────────────────────────────────
# 3 pre-zeroing values (0.0 → become NaN after Step 1 artifact cleaning),
# followed by 10 consecutive valid values. The first valid anchor is index 3;
# its 20-second look-ahead (arr[3:23], but array ends at index 12) contains
# 10 valid readings which equals window=10, so the function should return 3.
_test = np.array([0.0, 0.0, 0.0,                          # pre-zeroing artifact
                  75.0, 80.0, 70.0, 78.0, 72.0,           # valid — anchor at index 3
                  68.0, 74.0, 80.0, 76.0, 71.0])           # valid — 10 total valid values
_result = find_stable_start(_test)
assert _result == 3, (
    f'Expected 3, got {_result}. '
    f'Check that the anchor + density logic is working correctly.'
)
print('find_stable_start unit test passed (expected index 3).')

find_stable_start unit test passed (expected index 3).


In [15]:
# ── Process all ART_MBP episodes ───────────────────────────────────────────
#
# We load each patient's ART_MBP waveform ONCE and iterate through all of
# that patient's episodes using the same array (efficiency requirement).

art_results = []

art_episodes   = episodes[episodes['bp_source'] == 'ART_MBP'].copy()
n_art_patients = art_episodes['caseid'].nunique()
pt_idx         = 0

for caseid, group in art_episodes.groupby('caseid'):
    pt_idx += 1
    # Print progress every 50 patients so the cell output isn't overwhelming.
    if pt_idx == 1 or pt_idx % 50 == 0 or pt_idx == n_art_patients:
        print(f'[{pt_idx}/{n_art_patients}] caseid {caseid} ({len(group)} ep(s))...')

    # Load ART_MBP at 1-second intervals.
    # Returns shape (n_seconds, 1); index 0 = time 0 s from recording start.
    arr = vitaldb.load_case(caseid, [ART_TRACK], INTERVAL_SEC)

    if arr is None or len(arr) == 0:
        print(f'  WARNING: no ART_MBP data for caseid {caseid}')
        for _, ep in group.iterrows():
            art_results.append({
                'caseid': caseid, 'episode_number': int(ep['episode_number']),
                'episode_end_sec': ep['episode_end_sec'],
                'hypotension_label': np.nan, 'bp_outcome_min': np.nan,
                'bp_outcome_mean': np.nan, 'outcome_window_availability': 0.0,
                'bp_source': 'ART_MBP', 'nibp_outcome_imputed': False,
            })
        continue

    art     = arr[:, 0].copy()
    n_total = len(art)

    # ── Artifact cleaning ──
    # Step 1: remove physically impossible values.
    art[(art < ART_MIN) | (art > ART_MAX)] = np.nan

    # Step 2: remove pre-zeroing artifact.
    # NaN everything before the first 10 consecutive stable seconds.
    stable_idx = find_stable_start(art)
    if stable_idx > 0:
        art[:stable_idx] = np.nan

    # ── Label each episode ──
    for _, ep in group.iterrows():
        end_idx   = int(ep['episode_end_sec'])    # floor to whole second
        win_start = end_idx
        win_end   = min(n_total, end_idx + OUTCOME_WINDOW_SEC)

        if win_start >= n_total:
            # Episode end is beyond recording length — no data available.
            art_results.append({
                'caseid': caseid, 'episode_number': int(ep['episode_number']),
                'episode_end_sec': ep['episode_end_sec'],
                'hypotension_label': np.nan, 'bp_outcome_min': np.nan,
                'bp_outcome_mean': np.nan, 'outcome_window_availability': 0.0,
                'bp_source': 'ART_MBP', 'nibp_outcome_imputed': False,
            })
            continue

        window_vals = art[win_start:win_end]
        valid_vals  = window_vals[~np.isnan(window_vals)]
        n_valid     = len(valid_vals)

        # outcome_window_availability = fraction of 300 s with valid data.
        availability = n_valid / OUTCOME_WINDOW_SEC

        # ── Assign label ──
        if n_valid >= MIN_VALID_READINGS:
            if np.any(valid_vals < HYPO_THRESHOLD):
                label = 1
            else:
                label = 0
            min_val  = float(np.min(valid_vals))
            mean_val = float(np.mean(valid_vals))
        else:
            # Fewer than 10 valid readings — cannot reliably label.
            label    = np.nan
            min_val  = float(np.min(valid_vals)) if n_valid > 0 else np.nan
            mean_val = float(np.mean(valid_vals)) if n_valid > 0 else np.nan

        art_results.append({
            'caseid': caseid, 'episode_number': int(ep['episode_number']),
            'episode_end_sec': ep['episode_end_sec'],
            'hypotension_label': label,
            'bp_outcome_min': round(min_val, 2) if not np.isnan(min_val) else np.nan,
            'bp_outcome_mean': round(mean_val, 2) if not np.isnan(mean_val) else np.nan,
            'outcome_window_availability': round(availability, 4),
            'bp_source': 'ART_MBP', 'nibp_outcome_imputed': False,
        })

print(f'\nART_MBP processing complete: {len(art_results)} episode results')

[1/361] caseid 12 (4 ep(s))...
[50/361] caseid 1002 (2 ep(s))...
[100/361] caseid 2016 (4 ep(s))...
[150/361] caseid 2795 (1 ep(s))...
[200/361] caseid 3519 (1 ep(s))...
[250/361] caseid 4405 (1 ep(s))...
[300/361] caseid 5262 (6 ep(s))...
[350/361] caseid 6198 (2 ep(s))...
[361/361] caseid 6374 (9 ep(s))...

ART_MBP processing complete: 982 episode results


In [16]:
# ── Process all NIBP_estimated episodes ────────────────────────────────────
#
# Same one-load-per-patient approach as for ART_MBP.
# Special handling for the 2 nibp_outcome_imputed=True episodes:
#   - These had no NIBP readings in the primary window during the NIBP_data stage.
#   - Try the outcome window first (episode_end_sec + 300 s) — if it has data, use it.
#   - If the outcome window is ALSO empty, fall back to the expanded window
#     [episode_start_sec - 300 s, episode_start_sec + 600 s] to find the nearest
#     available reading (the 'imputed value' confirmed during NIBP_data processing).
#   - Imputed episodes CANNOT receive hypotension_label = NaN.

nibp_results   = []

nibp_episodes   = episodes[episodes['bp_source'] == 'NIBP_estimated'].copy()
n_nibp_patients = nibp_episodes['caseid'].nunique()
pt_idx          = 0

for caseid, group in nibp_episodes.groupby('caseid'):
    pt_idx += 1
    print(f'[{pt_idx}/{n_nibp_patients}] caseid {caseid} ({len(group)} ep(s))...')

    arr = vitaldb.load_case(caseid, [NIBP_SBP_TRACK, NIBP_DBP_TRACK], INTERVAL_SEC)

    if arr is None or len(arr) == 0:
        print(f'  WARNING: no NIBP data for caseid {caseid}')
        for _, ep in group.iterrows():
            nibp_results.append({
                'caseid': caseid, 'episode_number': int(ep['episode_number']),
                'episode_end_sec': ep['episode_end_sec'],
                'hypotension_label': np.nan, 'bp_outcome_min': np.nan,
                'bp_outcome_mean': np.nan, 'outcome_window_availability': 0.0,
                'bp_source': 'NIBP_estimated',
                'nibp_outcome_imputed': bool(ep['nibp_outcome_imputed']),
            })
        continue

    sbp = arr[:, 0].copy()
    dbp = arr[:, 1].copy()
    n_total = len(sbp)

    # ── Artifact cleaning (re-applied as safety check) ──
    sbp[(sbp < NIBP_SBP_MIN) | (sbp > NIBP_SBP_MAX)] = np.nan
    dbp[(dbp < NIBP_DBP_MIN) | (dbp > NIBP_DBP_MAX)] = np.nan
    bad_rel = sbp <= dbp
    sbp[bad_rel] = np.nan
    dbp[bad_rel] = np.nan
    est_map = dbp + (sbp - dbp) / 3
    est_map[(est_map < NIBP_MAP_MIN) | (est_map > NIBP_MAP_MAX)] = np.nan

    for _, ep in group.iterrows():
        end_idx    = int(ep['episode_end_sec'])
        start_idx  = int(ep['episode_start_sec'])   # used only for imputed fallback
        is_imputed = bool(ep['nibp_outcome_imputed'])

        # ── Outcome window ──
        win_start = end_idx
        win_end   = min(n_total, end_idx + OUTCOME_WINDOW_SEC)

        if win_start < n_total:
            window_vals  = est_map[win_start:win_end]
            valid_outcome = window_vals[~np.isnan(window_vals)]
        else:
            valid_outcome = np.array([], dtype=float)

        n_valid      = len(valid_outcome)
        # availability always reflects the actual outcome window
        availability = n_valid / OUTCOME_WINDOW_SEC

        # ── Determine which values to use for labeling ──
        if is_imputed:
            if n_valid >= 1:
                # Outcome window has data — use it directly.
                use_vals = valid_outcome
            else:
                # Outcome window is empty.
                # Fall back to the expanded window that was used in the NIBP_data
                # stage to confirm a reading exists (centered on episode_start_sec).
                fb_start = max(0, start_idx - NIBP_EXPANDED_BEFORE)
                fb_end   = min(n_total, start_idx + NIBP_EXPANDED_AFTER)
                fb_vals  = est_map[fb_start:fb_end]
                use_vals = fb_vals[~np.isnan(fb_vals)]
                print(f'  caseid {caseid} ep {int(ep["episode_number"])}: '
                      f'outcome window empty, using fallback ({len(use_vals)} readings)')

            # Imputed episodes: ALWAYS assign 0 or 1 (never NaN).
            # The imputed MAP value is the mean of the available readings.
            imputed_map  = float(np.mean(use_vals))
            label        = 1 if imputed_map < HYPO_THRESHOLD else 0
            min_val      = float(np.min(use_vals))
            mean_val     = imputed_map

        else:
            # ── Non-imputed NIBP episode ──
            if n_valid >= MIN_VALID_READINGS:
                if np.any(valid_outcome < HYPO_THRESHOLD):
                    label = 1
                else:
                    label = 0
                min_val  = float(np.min(valid_outcome))
                mean_val = float(np.mean(valid_outcome))
            else:
                # Fewer than 10 valid values → unreliable, set NaN.
                label    = np.nan
                min_val  = float(np.min(valid_outcome)) if n_valid > 0 else np.nan
                mean_val = float(np.mean(valid_outcome)) if n_valid > 0 else np.nan

        nibp_results.append({
            'caseid': caseid, 'episode_number': int(ep['episode_number']),
            'episode_end_sec': ep['episode_end_sec'],
            'hypotension_label': label,
            'bp_outcome_min': round(min_val, 2) if not np.isnan(min_val) else np.nan,
            'bp_outcome_mean': round(mean_val, 2) if not np.isnan(mean_val) else np.nan,
            'outcome_window_availability': round(availability, 4),
            'bp_source': 'NIBP_estimated',
            'nibp_outcome_imputed': is_imputed,
        })

print(f'\nNIBP processing complete: {len(nibp_results)} episode results')

[1/96] caseid 42 (1 ep(s))...
[2/96] caseid 174 (2 ep(s))...
[3/96] caseid 212 (2 ep(s))...
[4/96] caseid 253 (2 ep(s))...
[5/96] caseid 257 (3 ep(s))...
[6/96] caseid 285 (3 ep(s))...
[7/96] caseid 365 (1 ep(s))...
[8/96] caseid 539 (2 ep(s))...
  caseid 539 ep 1: outcome window empty, using fallback (116 readings)
  caseid 539 ep 2: outcome window empty, using fallback (81 readings)
[9/96] caseid 554 (1 ep(s))...
[10/96] caseid 569 (8 ep(s))...
[11/96] caseid 581 (3 ep(s))...
[12/96] caseid 643 (6 ep(s))...
[13/96] caseid 696 (2 ep(s))...
[14/96] caseid 708 (4 ep(s))...
[15/96] caseid 713 (1 ep(s))...
[16/96] caseid 862 (2 ep(s))...
[17/96] caseid 884 (1 ep(s))...
[18/96] caseid 1072 (3 ep(s))...
[19/96] caseid 1110 (3 ep(s))...
[20/96] caseid 1269 (6 ep(s))...
[21/96] caseid 1276 (2 ep(s))...
[22/96] caseid 1377 (1 ep(s))...
[23/96] caseid 1378 (3 ep(s))...
[24/96] caseid 1481 (1 ep(s))...
[25/96] caseid 1607 (6 ep(s))...
[26/96] caseid 1622 (3 ep(s))...
[27/96] caseid 1626 (7 ep(s)

In [17]:
# Combine ART_MBP and NIBP results into a single DataFrame.
results_df = pd.concat(
    [pd.DataFrame(art_results), pd.DataFrame(nibp_results)],
    ignore_index=True
)

# Sort by caseid then episode_number to match the input file order.
results_df = results_df.sort_values(['caseid', 'episode_number']).reset_index(drop=True)

# Enforce the exact output column order specified in the spec.
col_order = [
    'caseid', 'episode_number', 'episode_end_sec', 'hypotension_label',
    'bp_outcome_min', 'bp_outcome_mean', 'outcome_window_availability',
    'bp_source', 'nibp_outcome_imputed',
]
results_df = results_df[col_order]

print(f'Combined results: {len(results_df)} episodes')
print(f'Columns: {results_df.columns.tolist()}')
print()
print(results_df.head(5).to_string())

Combined results: 1284 episodes
Columns: ['caseid', 'episode_number', 'episode_end_sec', 'hypotension_label', 'bp_outcome_min', 'bp_outcome_mean', 'outcome_window_availability', 'bp_source', 'nibp_outcome_imputed']

   caseid  episode_number  episode_end_sec  hypotension_label  bp_outcome_min  bp_outcome_mean  outcome_window_availability bp_source  nibp_outcome_imputed
0      12               1      8746.802778                1.0            49.0            61.31                          0.5   ART_MBP                 False
1      12               2      8918.786111                1.0            47.0            55.15                          0.5   ART_MBP                 False
2      12               3      9223.980556                1.0            56.0            62.10                          0.5   ART_MBP                 False
3      12               4      9641.541667                0.0            65.0            70.27                          0.5   ART_MBP                 False
4   

In [18]:
# ── CHECK 1: Shape must match arrhythmia_episodes_updated.csv exactly ───────
print('=' * 65)
print('CHECK 1: Row count')
print('=' * 65)

expected_n = len(episodes)
actual_n   = len(results_df)

print(f'Expected rows : {expected_n}')
print(f'Actual rows   : {actual_n}')

if actual_n == expected_n:
    print('PASS: Row counts match.')
else:
    print(f'FAIL: {abs(actual_n - expected_n)} row(s) differ.')
    # Identify which episodes are missing from results
    input_keys   = set(zip(episodes['caseid'], episodes['episode_number']))
    output_keys  = set(zip(results_df['caseid'], results_df['episode_number']))
    missing      = input_keys - output_keys
    extra        = output_keys - input_keys
    if missing:
        print(f'  Missing episodes ({len(missing)}):  {sorted(missing)[:20]}')
    if extra:
        print(f'  Extra episodes  ({len(extra)}):  {sorted(extra)[:20]}')

CHECK 1: Row count
Expected rows : 1284
Actual rows   : 1284
PASS: Row counts match.


In [19]:
# ── CHECK 2: Label counts and hypotension rates ─────────────────────────────
print('=' * 65)
print('CHECK 2: Label distribution')
print('=' * 65)

n_pos  = int((results_df['hypotension_label'] == 1).sum())
n_neg  = int((results_df['hypotension_label'] == 0).sum())
n_nan  = int(results_df['hypotension_label'].isna().sum())
n_lab  = n_pos + n_neg
rate   = 100 * n_pos / n_lab if n_lab > 0 else float('nan')

print(f'hypotension_label = 1  : {n_pos}')
print(f'hypotension_label = 0  : {n_neg}')
print(f'hypotension_label = NaN: {n_nan}')
print(f'Hypotension rate       : {rate:.1f}%  (of labeled episodes)')

print()
print('--- By bp_source ---')

for src in ['ART_MBP', 'NIBP_estimated']:
    sub   = results_df[results_df['bp_source'] == src]
    s_pos = int((sub['hypotension_label'] == 1).sum())
    s_neg = int((sub['hypotension_label'] == 0).sum())
    s_nan = int(sub['hypotension_label'].isna().sum())
    s_lab = s_pos + s_neg
    s_rate = 100 * s_pos / s_lab if s_lab > 0 else float('nan')
    print(f'  {src}: label=1 {s_pos}, label=0 {s_neg}, NaN {s_nan}, '
          f'rate {s_rate:.1f}%')

# Flag large discrepancy between ART and NIBP rates
art_sub  = results_df[results_df['bp_source'] == 'ART_MBP']
nibp_sub = results_df[results_df['bp_source'] == 'NIBP_estimated']
art_rate  = 100 * (art_sub['hypotension_label']==1).sum() / max(1,(art_sub['hypotension_label'].isin([0,1])).sum())
nibp_rate = 100 * (nibp_sub['hypotension_label']==1).sum() / max(1,(nibp_sub['hypotension_label'].isin([0,1])).sum())

if abs(art_rate - nibp_rate) > 20:
    print()
    print(f'*** FLAG: ART rate ({art_rate:.1f}%) and NIBP rate ({nibp_rate:.1f}%) '
          f'differ by >{abs(art_rate-nibp_rate):.1f}pp.')
    print('    This may indicate systematic label quality differences between')
    print('    continuous (ART) and intermittent (NIBP) monitoring.')

CHECK 2: Label distribution
hypotension_label = 1  : 446
hypotension_label = 0  : 750
hypotension_label = NaN: 88
Hypotension rate       : 37.3%  (of labeled episodes)

--- By bp_source ---
  ART_MBP: label=1 388, label=0 510, NaN 84, rate 43.2%
  NIBP_estimated: label=1 58, label=0 240, NaN 4, rate 19.5%

*** FLAG: ART rate (43.2%) and NIBP rate (19.5%) differ by >23.7pp.
    This may indicate systematic label quality differences between
    continuous (ART) and intermittent (NIBP) monitoring.


In [20]:
# ── CHECK 3: Distribution of bp_outcome_min ─────────────────────────────────
print('=' * 65)
print('CHECK 3: bp_outcome_min distribution (labeled episodes only)')
print('=' * 65)

labeled = results_df[results_df['hypotension_label'].notna()].copy()
mins     = labeled['bp_outcome_min'].dropna()

print(f'n (labeled with min available): {len(mins)}')
print(f'Mean   : {mins.mean():.1f} mmHg')
print(f'Median : {mins.median():.1f} mmHg')
print(f'Min    : {mins.min():.1f} mmHg')
print(f'Max    : {mins.max():.1f} mmHg')

borderline = int(((mins >= 60) & (mins < 65)).sum())
severe     = int((mins < 50).sum())
print()
print(f'Episodes where bp_outcome_min is 60-65 mmHg (borderline, just missed threshold): {borderline}')
print(f'Episodes where bp_outcome_min < 50 mmHg   (severe hypotension):                  {severe}')

print()
print('--- bp_outcome_min histogram (rounded to nearest 5 mmHg) ---')
hist = mins.round(-1 if False else 0)   # round to integer mmHg
bins = [40, 50, 55, 60, 65, 70, 80, 90, 100, 110, 120, 130, 140, 150, 181]
for lo, hi in zip(bins[:-1], bins[1:]):
    count = int(((mins >= lo) & (mins < hi)).sum())
    bar   = '#' * (count // 5)
    print(f'  [{lo:3d}-{hi:3d}) : {count:4d}  {bar}')

CHECK 3: bp_outcome_min distribution (labeled episodes only)
n (labeled with min available): 1196
Mean   : 70.9 mmHg
Median : 70.0 mmHg
Min    : 40.0 mmHg
Max    : 143.7 mmHg

Episodes where bp_outcome_min is 60-65 mmHg (borderline, just missed threshold): 140
Episodes where bp_outcome_min < 50 mmHg   (severe hypotension):                  114

--- bp_outcome_min histogram (rounded to nearest 5 mmHg) ---
  [ 40- 50) :  114  ######################
  [ 50- 55) :   70  ##############
  [ 55- 60) :  122  ########################
  [ 60- 65) :  140  ############################
  [ 65- 70) :  149  #############################
  [ 70- 80) :  274  ######################################################
  [ 80- 90) :  172  ##################################
  [ 90-100) :   96  ###################
  [100-110) :   29  #####
  [110-120) :   25  #####
  [120-130) :    3  
  [130-140) :    1  
  [140-150) :    1  
  [150-181) :    0  


In [21]:
# ── CHECK 4: outcome_window_availability by bp_source ───────────────────────
print('=' * 65)
print('CHECK 4: Mean outcome_window_availability by bp_source')
print('=' * 65)

for src in ['ART_MBP', 'NIBP_estimated']:
    sub  = results_df[results_df['bp_source'] == src]['outcome_window_availability']
    print(f'{src}:')
    print(f'  Mean   : {sub.mean():.3f}')
    print(f'  Median : {sub.median():.3f}')
    print(f'  Min    : {sub.min():.3f}')
    print(f'  Max    : {sub.max():.3f}')

# Note for ART_MBP: expected >0.4 given ~1 Hz update rate on 1 Hz grid.
# Note for NIBP: cuff inflates intermittently, so availability may be low.
art_avail = results_df[results_df['bp_source'] == 'ART_MBP']['outcome_window_availability'].mean()
if art_avail < 0.4:
    print()
    print(f'*** FLAG: ART_MBP mean availability ({art_avail:.3f}) is below expected 0.4.')
    print('    Possible causes: episodes near end of recording, or heavy artifact removal.')

CHECK 4: Mean outcome_window_availability by bp_source
ART_MBP:
  Mean   : 0.437
  Median : 0.500
  Min    : 0.000
  Max    : 0.503
NIBP_estimated:
  Mean   : 0.468
  Median : 0.500
  Min    : 0.000
  Max    : 0.500


In [22]:
# ── CHECK 5: Temporal alignment — 3 hypotension episodes ────────────────────
#
# For 3 randomly selected episodes with hypotension_label = 1, reload the BP
# waveform and print the raw values over the first 60 seconds of the outcome
# window. Confirms that the drop below 65 mmHg actually occurs AFTER
# episode_end_sec (not before, which would indicate a window offset bug).
print('=' * 65)
print('CHECK 5: Temporal alignment (first 60 s of outcome window, 3 hypotension episodes)')
print('=' * 65)

ioh_eps = results_df[results_df['hypotension_label'] == 1].copy()
random.seed(42)
n_sample = min(3, len(ioh_eps))
sample   = ioh_eps.sample(n_sample, random_state=42)

for _, row in sample.iterrows():
    caseid   = int(row['caseid'])
    ep_num   = int(row['episode_number'])
    end_sec  = int(row['episode_end_sec'])
    bp_src   = row['bp_source']

    print(f'\n--- caseid={caseid}, episode={ep_num}, '
          f'episode_end_sec={end_sec}s, bp_source={bp_src} ---')
    print(f'  bp_outcome_min recorded = {row["bp_outcome_min"]} mmHg')

    # Reload the waveform for this patient.
    if bp_src == 'ART_MBP':
        arr2 = vitaldb.load_case(caseid, [ART_TRACK], INTERVAL_SEC)
        bp2  = arr2[:, 0].copy()
        bp2[(bp2 < ART_MIN) | (bp2 > ART_MAX)] = np.nan
        bp2[:find_stable_start(bp2)] = np.nan
    else:
        arr2  = vitaldb.load_case(caseid, [NIBP_SBP_TRACK, NIBP_DBP_TRACK], INTERVAL_SEC)
        sbp2  = arr2[:, 0].copy()
        dbp2  = arr2[:, 1].copy()
        sbp2[(sbp2 < NIBP_SBP_MIN) | (sbp2 > NIBP_SBP_MAX)] = np.nan
        dbp2[(dbp2 < NIBP_DBP_MIN) | (dbp2 > NIBP_DBP_MAX)] = np.nan
        br2   = sbp2 <= dbp2
        sbp2[br2] = np.nan
        dbp2[br2] = np.nan
        bp2   = dbp2 + (sbp2 - dbp2) / 3
        bp2[(bp2 < NIBP_MAP_MIN) | (bp2 > NIBP_MAP_MAX)] = np.nan

    n2         = len(bp2)
    slice_end  = min(n2, end_sec + 60)      # first 60 seconds of outcome window
    window_60  = bp2[end_sec:slice_end]

    # Print only non-NaN values (and de-duplicate repeated NIBP broadcasts).
    rows_to_show = []
    prev_val = None
    for t in range(len(window_60)):
        v = window_60[t]
        if not np.isnan(v) and v != prev_val:
            rows_to_show.append({
                'abs_second': end_sec + t,
                'MAP_mmHg':   round(float(v), 1),
                'below_65':   v < 65,
            })
            prev_val = v

    if rows_to_show:
        disp = pd.DataFrame(rows_to_show)
        print(disp.to_string(index=False))
        first_hypo = next((r['abs_second'] for r in rows_to_show if r['below_65']), None)
        if first_hypo is not None:
            print(f'  --> First sub-65 value at second {first_hypo} '
                  f'({first_hypo - end_sec} s after episode_end_sec) [CORRECT: after end]')
    else:
        print('  (no non-NaN readings in first 60 s of outcome window)')

CHECK 5: Temporal alignment (first 60 s of outcome window, 3 hypotension episodes)

--- caseid=3930, episode=4, episode_end_sec=2610s, bp_source=ART_MBP ---
  bp_outcome_min recorded = 49.0 mmHg
 abs_second  MAP_mmHg  below_65
       2610      69.0     False
       2614      63.0      True
       2616      64.0      True
       2618      66.0     False
       2620      67.0     False
       2622      68.0     False
       2628      67.0     False
       2630      68.0     False
       2632      67.0     False
       2634      68.0     False
       2638      69.0     False
       2640      68.0     False
       2652      67.0     False
       2660      66.0     False
       2668      65.0     False
  --> First sub-65 value at second 2614 (4 s after episode_end_sec) [CORRECT: after end]

--- caseid=5115, episode=8, episode_end_sec=3351s, bp_source=NIBP_estimated ---
  bp_outcome_min recorded = 63.0 mmHg
 abs_second  MAP_mmHg  below_65
       3351      79.7     False

--- caseid=1996, epi

In [23]:
# ── CHECK 6: No label=1 episode can have bp_outcome_min >= 65 ───────────────
#
# This is a logical integrity check. If label=1, at least one reading was < 65,
# so the minimum MUST be < 65. Any violation is a labeling bug.
print('=' * 65)
print('CHECK 6: Logical integrity — label=1 implies bp_outcome_min < 65')
print('=' * 65)

violations = results_df[
    (results_df['hypotension_label'] == 1) &
    (results_df['bp_outcome_min'] >= HYPO_THRESHOLD)
]

if len(violations) == 0:
    print('PASS: No label=1 episode has bp_outcome_min >= 65 mmHg.')
else:
    print(f'FAIL: {len(violations)} labeling violation(s) found:')
    print(violations[['caseid', 'episode_number', 'hypotension_label', 'bp_outcome_min']].to_string(index=False))
    print('These must be fixed before saving.')

# ── FINAL SAVE ───────────────────────────────────────────────────────────────
print()
all_checks_pass = (len(results_df) == len(episodes)) and (len(violations) == 0)

if all_checks_pass:
    output_path = OUTPUT_DIR / 'episode_hypotension_labels.csv'
    results_df.to_csv(output_path, index=False)
    print(f'Saved: {output_path}')
    print(f'  Rows    : {len(results_df)}')
    print(f'  Columns : {results_df.columns.tolist()}')
    print()
    print('All critical checks passed. File is ready for the feature-engineering stage.')
else:
    print('NOT SAVED: one or more critical checks failed. Review output above.')

CHECK 6: Logical integrity — label=1 implies bp_outcome_min < 65
PASS: No label=1 episode has bp_outcome_min >= 65 mmHg.

Saved: episode_hypotension_labels.csv
  Rows    : 1284
  Columns : ['caseid', 'episode_number', 'episode_end_sec', 'hypotension_label', 'bp_outcome_min', 'bp_outcome_mean', 'outcome_window_availability', 'bp_source', 'nibp_outcome_imputed']

All critical checks passed. File is ready for the feature-engineering stage.
